# Feature Statistics and Visualization


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

pd.set_option("display.float_format", "{:.4f}".format)
if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

# Resolve project root for both execution modes:
# - launched from repo root
# - launched from notebooks/
cwd = Path.cwd().resolve()
if (cwd / "data").exists() and (cwd / "outputs").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "outputs").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not resolve project root containing data/ and outputs/")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "crsp_monthly_features.parquet"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "feature-analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"All outputs will be saved to: {OUTPUT_DIR}")
print(f"Using seaborn: {HAS_SEABORN}")


## 1. Load Data and Basic Info


In [ ]:
try:
    df = pd.read_parquet(DATA_PATH)
except (ImportError, ValueError) as err:
    csv_fallback = DATA_PATH.with_suffix(".csv")
    if not csv_fallback.exists():
        raise
    print(f"Parquet engine unavailable ({err}). Falling back to CSV: {csv_fallback}")
    df = pd.read_csv(csv_fallback)

df["date"] = pd.to_datetime(df["date"])

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Unique stocks (permno): {df['permno'].nunique():,}")
print("\nColumns:")
for c in df.columns:
    print(f"  {c}")


In [ ]:
# Identify the _cs feature columns and the label
CS_FEATURES = [c for c in df.columns if c.endswith("_cs")]
LABEL = "ret_fwd_1m"

if LABEL not in df.columns:
    raise KeyError(f"Label column '{LABEL}' not found in dataframe")

print(f"Cross-sectional features ({len(CS_FEATURES)}):")
for c in CS_FEATURES:
    print(f"  - {c}")


## 2. Summary Statistics


In [ ]:
summary_stats = df[CS_FEATURES + [LABEL]].describe().T
display(summary_stats)

summary_stats_path = OUTPUT_DIR / "summary_statistics.csv"
summary_stats.to_csv(summary_stats_path)
print(f"Saved -> {summary_stats_path}")


## 3. Missing Values Analysis


In [ ]:
missing = df[CS_FEATURES + [LABEL]].isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
display(missing_df)

missing_table_path = OUTPUT_DIR / "missing_values.csv"
missing_df.to_csv(missing_table_path)
print(f"Saved -> {missing_table_path}")

fig, ax = plt.subplots(figsize=(10, 6))
missing_pct.plot.barh(ax=ax, color="salmon")
ax.set_xlabel("Missing (%)")
ax.set_title("Missing Values by Feature")
ax.invert_yaxis()
plt.tight_layout()
missing_fig_path = OUTPUT_DIR / "missing_values.png"
fig.savefig(missing_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {missing_fig_path}")


## 4. Correlation Analysis


In [ ]:
corr = df[CS_FEATURES + [LABEL]].corr()

corr_table_path = OUTPUT_DIR / "correlation_matrix.csv"
corr.to_csv(corr_table_path)
print(f"Saved -> {corr_table_path}")

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

if HAS_SEABORN:
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        ax=ax,
        annot_kws={"size": 8},
    )
else:
    masked = np.ma.masked_where(mask, corr.values)
    im = ax.imshow(masked, cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
    ax.set_xticks(np.arange(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(np.arange(len(corr.index)))
    ax.set_yticklabels(corr.index)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            if not mask[i, j]:
                ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
corr_fig_path = OUTPUT_DIR / "correlation_matrix_heatmap.png"
fig.savefig(corr_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {corr_fig_path}")


In [ ]:
ic = df[CS_FEATURES + [LABEL]].corr()[LABEL].drop(LABEL).sort_values(ascending=False)
ic_df = ic.rename("pearson_corr").to_frame()
display(ic_df)

ic_table_path = OUTPUT_DIR / "feature_label_correlation.csv"
ic_df.to_csv(ic_table_path)
print(f"Saved -> {ic_table_path}")

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["steelblue" if v > 0 else "salmon" for v in ic.values]
ic.plot.barh(ax=ax, color=colors)
ax.set_xlabel(f"Pearson correlation with {LABEL}")
ax.set_title("Feature-Label Correlation (pooled IC)")
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
ic_fig_path = OUTPUT_DIR / "feature_label_correlation.png"
fig.savefig(ic_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {ic_fig_path}")


## 5. Time-Series Diagnostics


In [ ]:
n_features = len(CS_FEATURES)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))

monthly_means = df.groupby("date")[CS_FEATURES].mean()
monthly_means_path = OUTPUT_DIR / "monthly_feature_means.csv"
monthly_means.to_csv(monthly_means_path)
print(f"Saved -> {monthly_means_path}")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(CS_FEATURES):
    ax = axes[i]
    ax.plot(monthly_means.index, monthly_means[col], linewidth=0.8, color="steelblue")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_title(col, fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=8)

for j in range(len(CS_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Cross-Sectional Mean of Each Feature Over Time", fontsize=14, y=1.01)
plt.tight_layout()
monthly_means_fig_path = OUTPUT_DIR / "monthly_feature_means.png"
fig.savefig(monthly_means_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {monthly_means_fig_path}")


In [ ]:
stock_count = df.groupby("date")["permno"].nunique().rename("stock_count")
stock_count_path = OUTPUT_DIR / "monthly_stock_count.csv"
stock_count.to_csv(stock_count_path)
print(f"Saved -> {stock_count_path}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(stock_count.index, stock_count.values, alpha=0.4, color="steelblue")
ax.plot(stock_count.index, stock_count.values, linewidth=0.8, color="steelblue")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Stocks")
ax.set_title("Number of Unique Stocks per Month")
plt.tight_layout()
stock_count_fig_path = OUTPUT_DIR / "monthly_stock_count.png"
fig.savefig(stock_count_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {stock_count_fig_path}")


## 6. Label Diagnostics


In [ ]:
label_data = df[LABEL].dropna()
monthly_label = df.groupby("date")[LABEL].mean().rename("monthly_mean")

label_summary = pd.DataFrame({
    "count": [label_data.shape[0]],
    "mean": [label_data.mean()],
    "median": [label_data.median()],
    "std": [label_data.std()],
    "min": [label_data.min()],
    "max": [label_data.max()],
})
display(label_summary)

label_summary_path = OUTPUT_DIR / "label_summary.csv"
label_summary.to_csv(label_summary_path, index=False)
print(f"Saved -> {label_summary_path}")

monthly_label_path = OUTPUT_DIR / "monthly_label_mean.csv"
monthly_label.to_csv(monthly_label_path)
print(f"Saved -> {monthly_label_path}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(label_data, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(label_data.mean(), color="red", linestyle="--", label=f"mean={label_data.mean():.4f}")
axes[0].axvline(label_data.median(), color="orange", linestyle="--", label=f"median={label_data.median():.4f}")
axes[0].set_title(f"{LABEL} Distribution")
axes[0].legend()

axes[1].bar(monthly_label.index, monthly_label.values, width=25, color="steelblue", alpha=0.7)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_title(f"Monthly Mean of {LABEL}")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
label_fig_path = OUTPUT_DIR / "label_diagnostics.png"
fig.savefig(label_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {label_fig_path}")
